In [2]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    cross_val_score
)

import sys
import os

sys.path.append(
    os.path.abspath("..")
)

from src.evaluate import rmse_cv

In [3]:
X_train = pd.read_csv(
    "../data/X_train_tree.csv"
)

X_test = pd.read_csv(
    "../data/X_test_tree.csv"
)

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

test = pd.read_csv(
    "../data/test.csv"
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)

(1458, 318)
(1459, 318)
(1458,)


In [4]:
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb_rmse = rmse_cv(
    xgb_model,
    X_train,
    y_train
).mean()

print(
    f"XGBoost RMSE: {xgb_rmse:.5f}"
)

XGBoost RMSE: 0.11697


In [5]:
xgb_model_v2 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v2 = rmse_cv(
    xgb_model_v2,
    X_train,
    y_train
).mean()

print(
    f"XGB v2 RMSE: {xgb_rmse_v2:.5f}"
)

XGB v2 RMSE: 0.11252


In [6]:
xgb_model_v2.fit(
    X_train,
    y_train
)

print("XGB v2 訓練完成")

XGB v2 訓練完成


In [7]:
xgb_pred_log_v2 = xgb_model_v2.predict(
    X_test
)

xgb_pred_v2 = np.expm1(
    xgb_pred_log_v2
)

print(xgb_pred_v2[:5])

[123594.875 160196.14  181534.1   190209.86  180845.84 ]


In [8]:
submission_v2 = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_pred_v2
})

submission_v2.to_csv(
    "../submissions/xgb_v2_submission.csv",
    index=False
)

print("XGB v2 Submission 已建立")

XGB v2 Submission 已建立


In [9]:
xgb_best = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)

xgb_best.fit(
    X_train,
    y_train
)

print("最佳 XGB 訓練完成")

xgb_best_pred_log = xgb_best.predict(
    X_test
)

xgb_best_pred = np.expm1(
    xgb_best_pred_log
)

submission_best = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_best_pred
})

submission_best.to_csv(
    "../submissions/xgb_random_search_submission.csv",
    index=False
)

xgb_best_rmse = rmse_cv(
    xgb_best,
    X_train,
    y_train
).mean()

print(
    f"XGB Random Search RMSE: {xgb_best_rmse:.5f}"
)

print("XGB Random Search Submission 已建立")

最佳 XGB 訓練完成
XGB Random Search RMSE: 0.11198
XGB Random Search Submission 已建立
